# Multivariate segmentation

`SharedBoundaryMultivariateSegmenter` performs a single segmentation across
channels (useful when all channels share the same change-points).
`IndependentMultivariateSegmenter` fits each channel independently.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bayesbreak import (
    BayesBreakGaussian,
    IndependentMultivariateSegmenter,
    SharedBoundaryMultivariateSegmenter,
)

rng = np.random.default_rng(123)
segment_lengths = [50, 70, 80]
n = sum(segment_lengths)
channel_means = [[0.0, 2.0, -1.0], [1.0, -1.0, 3.0], [2.0, 0.0, 1.0]]

Y = np.zeros((n, len(channel_means)))
for c, means in enumerate(channel_means):
    sig = np.concatenate([np.full(L, mu) for mu, L in zip(means, segment_lengths)])
    Y[:, c] = sig + 0.3 * rng.standard_normal(n)
X = np.arange(n).reshape(-1, 1)
print('Y shape:', Y.shape)

## Shared-boundary fit

In [ ]:
base = BayesBreakGaussian(k_max=10)
shared = SharedBoundaryMultivariateSegmenter(base).fit(X, Y)
print('k_map         :', shared.k_map_)
print('MAP boundaries:', shared.map_boundaries_)

In [ ]:
Y_hat = shared.predict(X)
fig, axes = plt.subplots(Y.shape[1], 1, figsize=(8, 5), sharex=True)
for c, ax in enumerate(axes):
    ax.plot(Y[:, c], 'o', ms=2, alpha=0.3, color='#888')
    ax.plot(Y_hat[:, c], lw=2, color='#4477AA')
    for b in shared.map_boundaries_[1:-1]:
        ax.axvline(b, color='#EE6677', ls=':')
    ax.set_ylabel(f'Channel {c}')
axes[-1].set_xlabel('index')
plt.show()

## Independent vs shared

In [ ]:
indep = IndependentMultivariateSegmenter(base).fit(X, Y)
for c, est in enumerate(indep.channel_estimators_):
    print(f'Channel {c}: indep MAP boundaries = {est.map_boundaries_}')
print('Shared MAP boundaries:', shared.map_boundaries_)